# Grocery Delivery Analytics Pipeline - 03 Gold and EDA

This notebook creates business-ready Gold tables, verifies revenue calculations at the order level, performs exploratory analysis, and persists every dataset required by the Databricks dashboard.

## 1. Setup and load Silver tables

In [ ]:
from pyspark.sql import functions as F
import plotly.express as px

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

def silver(name):
    return spark.table(f"{CATALOG}.{SCHEMA}.{name}_silver")

def write_gold(df, name):
    full_name = f"{CATALOG}.{SCHEMA}.{name}_gold"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full_name))
    print(f"{full_name}: {df.count():,} rows")

customers = silver("customers")
stores = silver("stores")
products = silver("products")
orders = silver("orders")
items = silver("order_items")
deliveries = silver("deliveries")

## 2. Enriched sales line Gold table

Owner contributions: Omar defines revenue fields; Maheshwar supplies product/category attributes; Sweta supplies order calendar/status attributes; Mannan supplies store attributes; Hazim supplies customer loyalty; Shreyansh supplies delivery outcomes.

In [ ]:
sales_line_gold = (
    items
    .join(products.select("ProductID", "ProductName", "Category", "UnitCost"), "ProductID", "inner")
    .join(orders.select(
        "OrderID", "CustomerID", "StoreID", "OrderTimestamp", "OrderDate", "YearMonth",
        "DayName", "IsWeekend", "OrderStatus", "PaymentMethod"
    ), "OrderID", "inner")
    .join(stores.select("StoreID", "StoreName", F.col("City").alias("StoreCity"), "StoreType"), "StoreID", "inner")
    .join(customers.select("CustomerID", "CustomerFullName", "LoyaltyStatus"), "CustomerID", "inner")
    .join(deliveries.select(
        "OrderID", "DriverID", "DistanceKm", "PromisedMinutes", "ActualMinutes", "DelayMinutes", "IsOnTime"
    ), "OrderID", "left")
    .filter(F.col("OrderStatus") == "Completed")
    .withColumn("GrossRevenue", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .withColumn("DiscountAmount", F.round(F.col("GrossRevenue") * F.col("DiscountPct"), 2))
    .withColumn("NetRevenue", F.round(F.col("GrossRevenue") - F.col("DiscountAmount"), 2))
    .withColumn("EstimatedCost", F.round(F.col("Quantity") * F.col("UnitCost"), 2))
    .withColumn("GrossProfit", F.round(F.col("NetRevenue") - F.col("EstimatedCost"), 2))
    .withColumn("ProfitMarginPct", F.round(F.col("GrossProfit") / F.col("NetRevenue") * 100, 2))
)
write_gold(sales_line_gold, "sales_line")
display(sales_line_gold.limit(10))

## 3. Order Gold table and correct average-order-value grain

Revenue is first summed to one row per order. Averages calculated later therefore represent true order values rather than line-item averages.

In [ ]:
order_gold = (
    sales_line_gold.groupBy(
        "OrderID", "CustomerID", "StoreID", "StoreName", "OrderTimestamp", "OrderDate",
        "YearMonth", "DayName", "IsWeekend", "PaymentMethod", "LoyaltyStatus"
    )
    .agg(
        F.round(F.sum("GrossRevenue"), 2).alias("GrossRevenue"),
        F.round(F.sum("DiscountAmount"), 2).alias("DiscountAmount"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.round(F.sum("GrossProfit"), 2).alias("GrossProfit"),
        F.sum("Quantity").alias("TotalItems"),
        F.first("DistanceKm", ignorenulls=True).alias("DistanceKm"),
        F.first("PromisedMinutes", ignorenulls=True).alias("PromisedMinutes"),
        F.first("ActualMinutes", ignorenulls=True).alias("ActualMinutes"),
        F.first("DelayMinutes", ignorenulls=True).alias("DelayMinutes"),
        F.first("IsOnTime", ignorenulls=True).alias("IsOnTime"),
    )
)
write_gold(order_gold, "orders")
display(order_gold.describe(["NetRevenue", "TotalItems", "ActualMinutes", "DelayMinutes"]))

## 4. Store performance - owner: Mannan

In [ ]:
store_performance_gold = (
    order_gold.groupBy("StoreID", "StoreName")
    .agg(
        F.countDistinct("OrderID").alias("CompletedOrders"),
        F.countDistinct("CustomerID").alias("UniqueCustomers"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
        F.round(F.avg("ActualMinutes"), 2).alias("AverageDeliveryMinutes"),
        F.round(F.avg(F.col("IsOnTime").cast("double")) * 100, 2).alias("OnTimeRatePct"),
    )
    .orderBy(F.desc("NetRevenue"))
)
write_gold(store_performance_gold, "store_performance")
display(store_performance_gold)

## 5. Product performance - owner: Maheshwar

In [ ]:
product_performance_gold = (
    sales_line_gold.groupBy("ProductID", "ProductName", "Category")
    .agg(
        F.sum("Quantity").alias("UnitsSold"),
        F.countDistinct("OrderID").alias("OrdersContainingProduct"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.round(F.sum("GrossProfit"), 2).alias("GrossProfit"),
        F.round(F.avg("DiscountPct") * 100, 2).alias("AverageDiscountPct"),
    )
    .orderBy(F.desc("NetRevenue"))
)
write_gold(product_performance_gold, "product_performance")
display(product_performance_gold.limit(15))

## 6. Customer behaviour - owner: Hazim Ali

In [ ]:
customer_behavior_gold = (
    order_gold.groupBy("CustomerID", "LoyaltyStatus")
    .agg(
        F.countDistinct("OrderID").alias("TotalOrders"),
        F.round(F.sum("NetRevenue"), 2).alias("TotalRevenue"),
        F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
        F.max("OrderDate").alias("LastOrderDate"),
    )
    .withColumn("IsRepeatCustomer", F.col("TotalOrders") >= 2)
)
write_gold(customer_behavior_gold, "customer_behavior")
display(customer_behavior_gold.orderBy(F.desc("TotalRevenue")).limit(10))

## 7. Dashboard summary tables - owners: Sweta, Omar Leopoldo, Shreyansh Pankaj

In [ ]:
monthly_revenue_gold = (
    order_gold.groupBy("YearMonth")
    .agg(
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.countDistinct("OrderID").alias("CompletedOrders"),
        F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
    ).orderBy("YearMonth")
)

category_performance_gold = (
    sales_line_gold.groupBy("Category")
    .agg(
        F.sum("Quantity").alias("UnitsSold"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.round(F.sum("GrossProfit"), 2).alias("GrossProfit"),
    ).orderBy(F.desc("NetRevenue"))
)

discount_impact_gold = (
    sales_line_gold
    .withColumn("DiscountBand", F.when(F.col("DiscountPct") == 0, "0%")
        .when(F.col("DiscountPct") <= 0.05, "5%")
        .when(F.col("DiscountPct") <= 0.10, "10%")
        .when(F.col("DiscountPct") <= 0.15, "15%")
        .otherwise("20%+"))
    .groupBy("DiscountBand")
    .agg(
        F.count("OrderItemID").alias("LineItems"),
        F.round(F.avg("Quantity"), 2).alias("AverageQuantity"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
    )
    .withColumn("SortOrder", F.when(F.col("DiscountBand") == "0%", 0)
        .when(F.col("DiscountBand") == "5%", 1)
        .when(F.col("DiscountBand") == "10%", 2)
        .when(F.col("DiscountBand") == "15%", 3).otherwise(4))
    .orderBy("SortOrder")
)

delivery_distance_gold = (
    order_gold.filter(F.col("ActualMinutes").isNotNull())
    .withColumn("DistanceBand", F.when(F.col("DistanceKm") <= 5, "0-5 km")
        .when(F.col("DistanceKm") <= 10, "5-10 km")
        .when(F.col("DistanceKm") <= 15, "10-15 km")
        .when(F.col("DistanceKm") <= 20, "15-20 km").otherwise("20+ km"))
    .groupBy("DistanceBand")
    .agg(
        F.countDistinct("OrderID").alias("Deliveries"),
        F.round(F.avg(F.col("IsOnTime").cast("double")) * 100, 2).alias("OnTimeRatePct"),
        F.round(F.avg("DelayMinutes"), 2).alias("AverageDelayMinutes"),
    )
    .withColumn("SortOrder", F.when(F.col("DistanceBand") == "0-5 km", 0)
        .when(F.col("DistanceBand") == "5-10 km", 1)
        .when(F.col("DistanceBand") == "10-15 km", 2)
        .when(F.col("DistanceBand") == "15-20 km", 3).otherwise(4))
    .orderBy("SortOrder")
)

loyalty_behavior_gold = (
    customer_behavior_gold.groupBy("LoyaltyStatus")
    .agg(
        F.countDistinct("CustomerID").alias("Customers"),
        F.round(F.avg("TotalOrders"), 2).alias("AverageOrders"),
        F.round(F.avg("AverageOrderValue"), 2).alias("AverageOrderValue"),
        F.round(F.avg(F.col("IsRepeatCustomer").cast("double")) * 100, 2).alias("RepeatRatePct"),
    )
)

for name, df in [
    ("monthly_revenue", monthly_revenue_gold), ("category_performance", category_performance_gold),
    ("discount_impact", discount_impact_gold), ("delivery_distance", delivery_distance_gold),
    ("loyalty_behavior", loyalty_behavior_gold),
]:
    write_gold(df, name)

## 8. KPI table and exploratory findings

In [ ]:
kpi_gold = order_gold.agg(
    F.round(F.sum("NetRevenue"), 2).alias("TotalNetRevenue"),
    F.countDistinct("OrderID").alias("CompletedOrders"),
    F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
    F.round(F.avg(F.col("IsOnTime").cast("double")) * 100, 2).alias("OnTimeDeliveryRatePct"),
).crossJoin(
    customer_behavior_gold.agg(
        F.round(F.avg(F.col("IsRepeatCustomer").cast("double")) * 100, 2).alias("RepeatCustomerRatePct")
    )
)
write_gold(kpi_gold, "kpi")
display(kpi_gold)

print("Summary statistics")
display(order_gold.describe(["NetRevenue", "GrossProfit", "TotalItems", "ActualMinutes", "DelayMinutes"]))
display(category_performance_gold)
display(discount_impact_gold)
display(delivery_distance_gold)
display(loyalty_behavior_gold.orderBy("LoyaltyStatus"))

## 9. EDA visualizations

In [ ]:
monthly_pdf = monthly_revenue_gold.toPandas()
fig = px.line(monthly_pdf, x="YearMonth", y="NetRevenue", markers=True,
              title="Monthly Net Revenue", labels={"NetRevenue": "Net Revenue ($)"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

category_pdf = category_performance_gold.toPandas()
px.bar(category_pdf, x="Category", y="NetRevenue", color="GrossProfit",
       title="Revenue and Profit by Product Category").show()

discount_pdf = discount_impact_gold.orderBy("SortOrder").toPandas()
px.bar(discount_pdf, x="DiscountBand", y="AverageQuantity",
       title="Average Quantity by Discount Level").show()

delivery_pdf = delivery_distance_gold.orderBy("SortOrder").toPandas()
px.line(delivery_pdf, x="DistanceBand", y="OnTimeRatePct", markers=True,
        title="On-Time Delivery Rate by Distance").show()

loyalty_pdf = loyalty_behavior_gold.toPandas()
px.bar(loyalty_pdf, x="LoyaltyStatus", y="RepeatRatePct", color="AverageOrderValue",
       title="Repeat-Customer Rate by Loyalty Tier").show()

## 10. Final validation

In [ ]:
required_gold = [
    "sales_line", "orders", "store_performance", "product_performance", "customer_behavior",
    "monthly_revenue", "category_performance", "discount_impact", "delivery_distance",
    "loyalty_behavior", "kpi",
]
for name in required_gold:
    full_name = f"{CATALOG}.{SCHEMA}.{name}_gold"
    assert spark.catalog.tableExists(full_name), f"Missing {full_name}"
    assert spark.table(full_name).count() > 0, f"Empty {full_name}"

assert order_gold.select("OrderID").distinct().count() == order_gold.count()
assert sales_line_gold.filter(F.col("NetRevenue") < 0).count() == 0
print("Gold/EDA validation passed. Use dashboard/dashboard_queries.sql to build the native dashboard.")